# Grammy Winners × Spotify ETL

This notebook follows three sequential stages:

1. **Normalize text:** load both datasets and standardize titles and artist credits.
2. **Record linkage:** generate blocked candidates, score title and artist similarity, and retain an auditable decision table.
3. **Finalize the merge:** combine exact matches with manually accepted candidates, validate counts, and export the final dataset.

Fuzzy scores are heuristic rankings, not calibrated probabilities. Automatic suggestions are kept separate from human decisions.

## Stage 1 — Text normalization

Load the cleaned Grammy winners and Spotify datasets, normalize titles and artist credits, and consolidate repeated Spotify track IDs before matching.

In [1]:
from pathlib import Path

import pandas as pd

# Resolve the project root from the notebook's current working directory.
project_root = next(
    (
        parent
        for parent in [Path.cwd(), *Path.cwd().parents]
        if (parent / "data" / "raw" / "spotify_dataset.csv").exists()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate data/raw/spotify_dataset.csv.")

grammy_path = next(
    iter(sorted(project_root.rglob("grammys_song_winners.csv"))),
    None,
)
if grammy_path is None:
    raise FileNotFoundError("Could not locate grammys_song_winners.csv in the project.")

spotify_raw = pd.read_csv(project_root / "data" / "raw" / "spotify_dataset.csv")
spotify_raw = spotify_raw.loc[:, ~spotify_raw.columns.str.match(r"^Unnamed")]
grammy_raw = pd.read_csv(grammy_path)

# Consolidate Spotify's repeated track IDs across genres.
spotify_tracks = (
    spotify_raw.groupby("track_id", as_index=False, sort=False)
    .agg(
        {
            **{
                column: "first"
                for column in spotify_raw.columns
                if column not in {"track_id", "track_genre"}
            },
            "track_genre": lambda values: "; ".join(
                pd.unique(values.dropna().astype(str))
            ),
        }
    )
)

grammy_records = grammy_raw.copy().reset_index(drop=True)
grammy_records["_grammy_id"] = range(len(grammy_records))

print("=== STAGE 1: DATA LOADING ===")
print(f"Grammy records: {len(grammy_records):,}")
print(f"Unique Spotify track IDs: {len(spotify_tracks):,}")

=== STAGE 1: DATA LOADING ===
Grammy records: 2,163
Unique Spotify track IDs: 89,741


## Stage 1 — Advanced text normalization

Normalize titles and artist credits separately. Common version suffixes are removed from titles, collaboration connectors are split for comparison, and known artist-name changes are handled through a configurable alias map. Keep the original source values unchanged for auditability.

The alias dictionary is intentionally explicit: add only verified identity changes there so unrelated artists are not merged accidentally.

In [2]:
import unicodedata

# Rebuild normalized keys with whitespace handling and version-aware title cleanup.
ARTIST_ALIASES = {
    "lady antebellum": "lady a",
    "lady a": "lady a",
}
TITLE_SUFFIXES = (
    " radio edit", " single version", " album version", " original version",
    " acoustic version", " bonus version", " remastered", " remaster",
    " version", " edit", " live",
)


def normalize_text(value):
    """Lowercase text, remove accents and punctuation, and collapse whitespace."""
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value).casefold())
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = "".join(char if char.isalnum() else " " for char in text)
    return " ".join(text.split())


def normalize_title(value):
    """Normalize titles after removing common release and collaboration suffixes."""
    if pd.isna(value):
        return ""
    text = str(value).casefold().strip()

    # Drop bracketed release metadata only when it contains a known version marker.
    for left, right in (("(", ")"), ("[", "]")):
        start = text.find(left)
        end = text.find(right, start + 1) if start >= 0 else -1
        if start >= 0 and end > start:
            bracket_text = text[start + 1:end]
            if any(marker in bracket_text for marker in ("live", "remaster", "radio edit", "version", "edit")):
                text = text[:start] + " " + text[end + 1:]

    for marker in (" featuring ", " feat. ", " feat ", " ft. ", " ft "):
        if marker in text:
            text = text.split(marker, 1)[0]
            break

    text = " ".join(text.replace("–", "-").replace("—", "-").split())
    changed = True
    while changed:
        changed = False
        for suffix in TITLE_SUFFIXES:
            if text.endswith(suffix) and len(text) > len(suffix):
                text = text[:-len(suffix)].rstrip(" -:.")
                changed = True
                break
        words = text.split()
        if words and words[-1].isdigit() and len(words[-1]) == 4:
            if len(words) > 1 and words[-2] in {"remaster", "remastered"}:
                text = " ".join(words[:-2])
                changed = True
    return normalize_text(text)


def split_artist_names(value):
    """Split featured/collaborating credits and canonicalize known artist aliases."""
    if pd.isna(value):
        return []
    text = str(value).casefold()
    text = text.replace("(with ", "|").replace(" with ", "|")
    for marker in ("featuring", "feat.", "feat", "ft.", "ft"):
        text = text.replace(f" {marker} ", "|")
    for separator in (";", ",", "&", "+", "(", ")"):
        text = text.replace(separator, "|")

    ignored = {"songwriter", "songwriters", "writer", "writers"}
    artist_keys = []
    for part in text.split("|"):
        key = normalize_text(part)
        key = ARTIST_ALIASES.get(key, key)
        if key and key not in ignored and key not in artist_keys:
            artist_keys.append(key)
    return artist_keys


grammy_records["_title_key"] = grammy_records["nominee"].map(normalize_title)
grammy_records["_artist_keys"] = grammy_records["artist_resolved"].map(split_artist_names)
spotify_tracks["_title_key"] = spotify_tracks["track_name"].map(normalize_title)
spotify_tracks["_artist_keys"] = spotify_tracks["artists"].map(split_artist_names)

print("Advanced title and artist normalization applied.")

Advanced title and artist normalization applied.


In [ ]:
from difflib import SequenceMatcher
import re

from rapidfuzz import fuzz

# -----------------------------------------------------------------------------
# STAGE 2: MODULAR RECORD LINKAGE
# -----------------------------------------------------------------------------
# This block creates the audit table that downstream cells expect. It keeps the
# original source fields intact and computes a weighted fuzzy score between
# Grammy titles/artists and Spotify tracks.
#
# Decision logic:
# - ACEPTAR_AUTOMATICO: high confidence on both title and artist, with a clear lead.
# - REVISION_MANUAL: plausible but uncertain, likely collaboration or alias variants.
# - RECHAZAR_AUTOMATICO: obvious mismatches and weak candidates.
#
# The output is intentionally auditable and conservative: it prefers recall while
# keeping dangerous false positives out of automatic acceptance.

project_root = Path.home() / "Projects" / "data-engineering" / "spotify-grammys-etl"
review_csv_path = project_root / "data" / "interim" / "spotify_grammy_match_review.csv"
review_csv_path.parent.mkdir(parents=True, exist_ok=True)

ARTIST_ALIASES = {
    "lady antebellum": "lady a",
    "lady a": "lady a",
    "diana ross and the supremes": "diana ross",
    "diana ross and the supremes featuring the miracles": "diana ross",
}

TITLE_CLEANUP_TOKENS = {
    "remastered",
    "remaster",
    "live",
    "radio edit",
    "single version",
    "album version",
    "original version",
    "acoustic version",
    "bonus version",
    "version",
    "edit",
}

ARTIST_IGNORE_TOKENS = {
    "songwriter",
    "songwriters",
    "writer",
    "writers",
    "band",
    "group",
    "artist",
}


def normalize_text(value):
    """Lowercase, strip accents, remove punctuation, and collapse whitespace."""
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value).casefold())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def normalize_title(value):
    """Normalize track titles while removing common release variants and extra metadata."""
    if pd.isna(value):
        return ""

    text = str(value).casefold().strip()
    text = text.replace("–", "-").replace("—", "-")

    # Remove bracketed metadata when it carries version/live markers.
    for left, right in (("(", ")"), ("[", "]")):
        start = text.find(left)
        end = text.find(right, start + 1) if start >= 0 else -1
        if start >= 0 and end > start:
            bracket_text = text[start + 1 : end].lower()
            if any(marker in bracket_text for marker in TITLE_CLEANUP_TOKENS):
                text = text[:start] + " " + text[end + 1 :]

    # Remove featuring/feat markers from the title itself, because the artist should carry that signal.
    for marker in (" featuring ", " feat. ", " feat ", " ft. ", " ft "):
        if marker in text:
            text = text.split(marker, 1)[0]
            break

    # Remove common release suffixes.
    changed = True
    while changed:
        changed = False
        for suffix in (
            " radio edit",
            " single version",
            " album version",
            " original version",
            " acoustic version",
            " bonus version",
            " remastered",
            " remaster",
            " version",
            " edit",
            " live",
        ):
            if text.endswith(suffix) and len(text) > len(suffix):
                text = text[: -len(suffix)].rstrip(" -:.")
                changed = True
                break

    # Drop trailing year after remaster/re-release labels.
    words = text.split()
    if words and words[-1].isdigit() and len(words[-1]) == 4:
        if len(words) > 1 and words[-2] in {"remaster", "remastered"}:
            text = " ".join(words[:-2])

    return normalize_text(text)


def split_artist_names(value):
    """Split collaborations into canonical artist tokens and apply known aliases."""
    if pd.isna(value):
        return []

    text = str(value).casefold()
    text = text.replace("(with ", "|").replace(" with ", "|")
    for marker in ("featuring", "feat.", "feat", "ft.", "ft"):
        text = text.replace(f" {marker} ", "|")

    for separator in (";", ",", "&", "+", "(", ")", "/"):
        text = text.replace(separator, "|")

    artist_keys = []
    for part in text.split("|"):
        key = normalize_text(part)
        key = ARTIST_ALIASES.get(key, key)
        if key and key not in ARTIST_IGNORE_TOKENS and key not in artist_keys:
            artist_keys.append(key)
    return artist_keys


def title_token_block(title_value):
    """Extract the meaningful title tokens used for blocking candidates."""
    normalized = normalize_title(title_value)
    if not normalized:
        return set()
    tokens = {token for token in normalized.split() if len(token) >= 3}
    return tokens


def score_title_similarity(title_a, title_b):
    """Title score: prefer token-aware matching and fall back to SequenceMatcher for edge cases."""
    a = normalize_title(title_a)
    b = normalize_title(title_b)

    if not a and not b:
        return 0.0
    if a == b:
        return 100.0
    if not a or not b:
        return 0.0

    score = fuzz.token_set_ratio(a, b)
    seq_score = SequenceMatcher(None, a.split(), b.split()).ratio() * 100
    return round(max(score, seq_score), 2)


def score_artist_similarity(artist_a, artist_b):
    """Artist score: compare canonical artist tokens while tolerating ordering and collaborations."""
    left = set(split_artist_names(artist_a))
    right = set(split_artist_names(artist_b))

    if not left and not right:
        return 0.0
    if left and right and left == right:
        return 100.0
    if not left or not right:
        return 0.0

    best_pair_score = 0.0
    for a_token in sorted(left):
        for b_token in sorted(right):
            best_pair_score = max(best_pair_score, fuzz.token_set_ratio(a_token, b_token))

    combined_left = " ".join(sorted(left))
    combined_right = " ".join(sorted(right))
    combined_score = fuzz.token_set_ratio(combined_left, combined_right)
    return round(max(best_pair_score, combined_score), 2)


def classify_linkage(puntaje_titulo, puntaje_artista, lead_gap):
    """Apply conservative thresholds for automatic acceptance, audit review, and rejection."""
    if puntaje_titulo >= 88 and puntaje_artista >= 80 and lead_gap >= 10:
        return "ACEPTAR_AUTOMATICO"
    if puntaje_titulo >= 72 and puntaje_artista >= 65:
        return "REVISION_MANUAL"
    return "RECHAZAR_AUTOMATICO"


# -----------------------------------------------------------------------------
# Build the blocked candidate set.
# -----------------------------------------------------------------------------
# We keep a bounded candidate list per Grammy record so the review table stays
# auditable and the search remains efficient. The key idea is: title tokens create
# blocks; within each block we score title and artist similarity separately.

spotify_tracks = spotify_tracks.copy()
grammy_records = grammy_records.copy()

spotify_tracks["_title_key"] = spotify_tracks["track_name"].map(normalize_title)
spotify_tracks["_artist_keys"] = spotify_tracks["artists"].map(split_artist_names)

grammy_records["_title_key"] = grammy_records["nominee"].map(normalize_title)
grammy_records["_artist_keys"] = grammy_records["artist_resolved"].map(split_artist_names)

candidate_rows = []

for _, grammy in grammy_records.iterrows():
    title_tokens = title_token_block(grammy["nominee"])
    if title_tokens:
        eligible = spotify_tracks[
            spotify_tracks["_title_key"].apply(
                lambda spotify_title: bool(title_tokens & set(spotify_title.split()))
            )
        ].copy()
    else:
        eligible = spotify_tracks.copy()

    if eligible.empty:
        continue

    for _, spotify in eligible.iterrows():
        puntaje_titulo = score_title_similarity(grammy["nominee"], spotify["track_name"])
        puntaje_artista = score_artist_similarity(grammy["artist_resolved"], spotify["artists"])
        score_linkage = round((0.40 * puntaje_titulo) + (0.60 * puntaje_artista), 2)

        candidate_rows.append(
            {
                "_grammy_id": int(grammy["_grammy_id"]),
                "year": grammy.get("year"),
                "category": grammy.get("category"),
                "nominee": grammy.get("nominee"),
                "artist_resolved": grammy.get("artist_resolved"),
                "track_id": spotify.get("track_id"),
                "track_name": spotify.get("track_name"),
                "artists": spotify.get("artists"),
                "puntaje_titulo": puntaje_titulo,
                "puntaje_artista": puntaje_artista,
                "score_linkage": score_linkage,
                "decision_manual": "PENDIENTE",
                "decision_sugerida": "",
                "notas_revision": "",
                "_title_key_grammy": normalize_title(grammy.get("nominee")),
                "_title_key_spotify": normalize_title(spotify.get("track_name")),
            }
        )

if not candidate_rows:
    linkage_candidates = pd.DataFrame(
        columns=[
            "_grammy_id",
            "year",
            "category",
            "nominee",
            "artist_resolved",
            "track_id",
            "track_name",
            "artists",
            "puntaje_titulo",
            "puntaje_artista",
            "score_linkage",
            "decision_manual",
            "decision_sugerida",
            "notas_revision",
            "_title_key_grammy",
            "_title_key_spotify",
        ]
    )
else:
    linkage_candidates = pd.DataFrame(candidate_rows)

# Scope candidate evaluation to the top 5 Spotify tracks per Grammy record to keep the audit table manageable.
linkage_candidates = linkage_candidates.sort_values(
    ["_grammy_id", "score_linkage", "puntaje_artista"],
    ascending=[True, False, False],
    na_position="last",
).drop_duplicates(subset=["_grammy_id", "track_id"], keep="first")
linkage_candidates = linkage_candidates.groupby("_grammy_id", group_keys=False).head(5).copy()

# Compute the lead over the second-best candidate to avoid ambiguous matches.
lead_gap = linkage_candidates.groupby("_grammy_id")["score_linkage"].transform(
    lambda values: (values.max() - values.nlargest(2).iloc[-1]) if len(values) >= 2 else values.max()
)
linkage_candidates["lead_gap"] = lead_gap
linkage_candidates["decision_sugerida"] = linkage_candidates.apply(
    lambda row: classify_linkage(
        row["puntaje_titulo"],
        row["puntaje_artista"],
        row["lead_gap"],
    ),
    axis=1,
)

# Keep only high-confidence positive and gray-zone candidates in the audit file.
# If a candidate is not accepted automatically, it remains in review for human validation.
linkage_candidates["decision_manual"] = "PENDIENTE"
linkage_candidates["notas_revision"] = linkage_candidates.apply(
    lambda row: "Matched by title/artist fuzzy linkage; review recommended when not auto-accepted." if row["decision_sugerida"] != "RECHAZAR_AUTOMATICO" else "Likely false positive; rejected by score thresholds.",
    axis=1,
)

# Ensure a consistent schema for downstream cells.
linkage_candidates = linkage_candidates[ 
    [
        "_grammy_id",
        "year",
        "category",
        "nominee",
        "artist_resolved",
        "track_id",
        "track_name",
        "artists",
        "puntaje_titulo",
        "puntaje_artista",
        "score_linkage",
        "lead_gap",
        "decision_manual",
        "decision_sugerida",
        "notas_revision",
    ]
].copy()

# Persist the audit table so the rest of the notebook can preserve manual decisions.
linkage_candidates.to_csv(review_csv_path, index=False)

print("=== STAGE 2: FUZZY LINKAGE ===")
print(f"Candidate rows generated: {len(linkage_candidates):,}")
print(f"Auto accepted: {(linkage_candidates['decision_sugerida'] == 'ACEPTAR_AUTOMATICO').sum():,}")
print(f"Manual review: {(linkage_candidates['decision_sugerida'] == 'REVISION_MANUAL').sum():,}")
print(f"Rejected: {(linkage_candidates['decision_sugerida'] == 'RECHAZAR_AUTOMATICO').sum():,}")
print(f"Saved review table: {review_csv_path}")

display(
    linkage_candidates.sort_values(["score_linkage", "puntaje_artista"], ascending=False)
    .head(15)
)


## Stage 2 — Blocked record linkage

The exact normalized title-and-artist join is retained as a high-confidence baseline. The fuzzy stage then builds candidate blocks from rare title tokens and character n-grams, scores title and artist credits independently, and stores the top candidates in an auditable CSV.

The combined score is 40% title similarity and 60% artist similarity. `ACEPTAR_AUTOMATICO` requires high scores on both fields plus a clear lead over the next candidate; `REVISION_MANUAL` captures plausible but uncertain pairs; weak candidates are labeled `RECHAZAR_AUTOMATICO`. These are heuristic scores, not calibrated probabilities. Review the thresholds against a labeled sample before relying on automatic decisions.

**Year caution:** Grammy award year and release year do not always match. Year is retained for traceability but is not used as a hard block.

In [3]:
from pathlib import Path
import pandas as pd

# Set the actual project root.
project_root = Path.home() / "Projects" / "data-engineering" / "spotify-grammys-etl"

# Define the manual review CSV path.
review_csv_path = (
    project_root / "data" / "interim" / "spotify_grammy_match_review.csv"
)

# Verify that the file exists.
print("Project root:", project_root)
print("Review file exists:", review_csv_path.exists())

# Load the review dataset.
review_final = pd.read_csv(review_csv_path)

print("Dataset shape:", review_final.shape)
print("Columns:", review_final.columns.tolist())

Project root: /home/sebastiancelorio.dev/Projects/data-engineering/spotify-grammys-etl
Review file exists: True
Dataset shape: (10470, 44)
Columns: ['year', 'title', 'category', 'nominee', 'artist_resolved', '_grammy_id', '_title_key', '_artist_keys', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', '_candidate_identity', 'criterio_candidato', 'artistas_comparados', 'puntaje_titulo', 'puntaje_artista', 'score_linkage', 'match_score', 'match_method', 'candidate_rank', 'score_margin', 'decision_sugerida', 'nota_automatica', 'decision_manual', 'notas_revision', 'review_id', 'decision_sugerida_linkage']


In [4]:
# ==========================================
# STAGE 2: EXACT TITLE AND ARTIST MATCHING
# ==========================================

exact_matches = grammy_records.merge(
    spotify_tracks,
    on="_title_key",
    how="inner",
    suffixes=("_grammy", "_spotify")
)

# Keep only records with at least one matching normalized artist.
exact_matches = exact_matches[
    exact_matches.apply(
        lambda row: bool(
            set(row["_artist_keys_grammy"])
            & set(row["_artist_keys_spotify"])
        ),
        axis=1
    )
].copy()
exact_grammy_ids = set(exact_matches["_grammy_id"].unique())

print("=== STAGE 2: EXACT MATCHES ===")
print(f"Exact matches found: {len(exact_matches):,}")
print(f"Unique Grammy records matched: {len(exact_grammy_ids):,}")
print(f"Unique Spotify tracks matched: {exact_matches['track_id'].nunique():,}")

display(
    exact_matches[
        ["year", "nominee", "artist_resolved",
         "track_name", "artists", "track_id"]
    ].head(10)
)

=== STAGE 2: EXACT MATCHES ===
Exact matches found: 385
Unique Grammy records matched: 241
Unique Spotify tracks matched: 301


,year,nominee,artist_resolved,track_name,artists,track_id
1,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,2Fxmhks0bxGSBdJ92vM42m
2,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,5KJaEu2ZMIt9FMvofqUgmg
3,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,154lbtHSyyOt0ao2QwbE4n
4,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,2yNKDG4FgwzQuL0JUEOunR
5,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,6DBdYfd4EkR6cUPSqtEwhQ
6,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,4Vzpf1TmxKmczOiTAfheBs
7,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,1c8h1SBYY44ys4EuaIZJMn
10,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,2Fxmhks0bxGSBdJ92vM42m
11,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,5KJaEu2ZMIt9FMvofqUgmg
12,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,154lbtHSyyOt0ao2QwbE4n


In [5]:
# Existing CSV fields may have been inferred as numeric when entirely empty.
linkage_candidates["notas_revision"] = linkage_candidates.get(
    "notas_revision", pd.Series("", index=linkage_candidates.index)
).fillna("").astype("object")

# Preserve existing human decisions when the same Grammy/Spotify pair is regenerated.
if review_csv_path.exists():
    previous_review = pd.read_csv(review_csv_path)
    identity_columns = ["year", "category", "nominee", "artist_resolved", "track_id"]
    if set(identity_columns + ["decision_manual"]).issubset(previous_review.columns):
        previous_review = previous_review.dropna(subset=["track_id"])
        for _, previous in previous_review.iterrows():
            match_mask = pd.Series(True, index=linkage_candidates.index)
            for column in identity_columns:
                match_mask &= linkage_candidates[column].astype(str).eq(str(previous[column]))
            if match_mask.any():
                decision = str(previous.get("decision_manual", "PENDIENTE"))
                if decision and decision.upper() != "NAN":
                    linkage_candidates.loc[match_mask, "decision_manual"] = decision
                if "notas_revision" in previous_review.columns:
                    linkage_candidates.loc[match_mask, "notas_revision"] = str(
                        previous.get("notas_revision", "") or ""
                    )

NameError: name 'linkage_candidates' is not defined

In [ ]:
# Diagnostic: best title score per Grammy record after advanced normalization.
diag = []
for _, grammy in grammy_records.iterrows():
    title_key = grammy["_title_key"]
    if not title_key:
        diag.append((grammy["_grammy_id"], "missing_title", 0.0))
        continue
    best = process.extractOne(title_key, spotify_titles, scorer=fuzz.token_set_ratio)
    diag.append((grammy["_grammy_id"], "ok", best[1] if best else 0.0))

diag = pd.DataFrame(diag, columns=["_grammy_id", "status", "best_title_score"])
print(diag["status"].value_counts())
print(diag["best_title_score"].describe())
print("Share with title score >= 85:", (diag["best_title_score"] >= 85).mean())

estado
ok    2163
Name: count, dtype: int64
count    2163.000000
mean       78.449128
std        15.775291
min        45.714286
25%        66.666667
50%        76.190476
75%       100.000000
max       100.000000
Name: mejor_title_score, dtype: float64
0.3296347665279704


In [ ]:
# Inspect highest-ranked candidates, including decision zones and audit details.
display(
    linkage_candidates.loc[linkage_candidates["track_id"].notna()]
    .sort_values(["score_linkage", "puntaje_artista"], ascending=False)
    [["year", "nominee", "artist_resolved", "track_name", "artists",
      "puntaje_titulo", "puntaje_artista", "score_linkage",
      "decision_sugerida", "decision_manual"]]
    .head(20)
)

,year,nominee,artist_resolved,track_name,artists,puntaje_titulo,puntaje_artista,score_linkage,decision_sugerida,decision_manual
0,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
1,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
2,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
3,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
4,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
5,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
6,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
7,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
8,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE
9,2019,Bad Guy,Billie Eilish,bad guy,Billie Eilish,100.0,100.0,100.0,REVISION_MANUAL,PENDIENTE


## Stage 3 — Final merge, validation, and export

Keep the normalized exact baseline, include fuzzy candidates suggested as `ACEPTAR_AUTOMATICO`, and include manual candidates only when `decision_manual` is `ACEPTAR`. Manual decisions are preserved in the audit file; all other fuzzy candidates remain excluded. Then validate duplicate Grammy–track pairs and export the merged and unmatched datasets.

In [ ]:
# Canonical Spotify feature columns retained in the final output.
spotify_output_columns = [
    column for column in spotify_tracks.columns
    if column not in {"_title_key", "_artist_keys"}
]

# Linkage IDs are already retained in the audit file; the final cell joins against these columns.


In [ ]:
# Read the persisted audit table so manual decisions remain the source of truth.
review_final = pd.read_csv(review_csv_path)
manual_accept = review_final["decision_manual"].fillna("PENDIENTE").str.upper().eq("ACEPTAR")
auto_accept = review_final["decision_sugerida"].fillna("").eq("ACEPTAR_AUTOMATICO")
accepted_review = review_final.loc[manual_accept | auto_accept].copy()
accepted_review["_accepted_automatically"] = auto_accept.loc[accepted_review.index]

# When distinct Spotify releases have identical title and artist credits, choose one
# representative instead of multiplying every Grammy record by duplicate catalog rows.
accepted_review = accepted_review.sort_values(
    ["_accepted_automatically", "score_linkage", "popularity"],
    ascending=[False, False, False],
    na_position="last",
).drop_duplicates(["_grammy_id", "track_id"], keep="first")

# Exact matches are confirmed by the normalized title + credited artist baseline.
exact_final = exact_matches.drop(
    columns=["_title_key", "_artist_keys"], errors="ignore"
).copy()
exact_final["match_method"] = "EXACT_TITLE_AND_ARTIST"

# New audit rows carry the Grammy ID; recover it by business keys for legacy rows only.
grammy_identity_columns = [
    "_grammy_id", "year", "category", "nominee", "artist_resolved"
]
accepted_with_grammy = accepted_review.copy()
accepted_with_grammy["_grammy_id"] = pd.to_numeric(
    accepted_with_grammy["_grammy_id"], errors="coerce"
)
missing_id = accepted_with_grammy["_grammy_id"].isna()
if missing_id.any():
    recovered_ids = accepted_with_grammy.loc[missing_id].drop(
        columns=["_grammy_id"], errors="ignore"
    ).merge(
        grammy_records[grammy_identity_columns],
        on=["year", "category", "nominee", "artist_resolved"],
        how="inner",
        validate="many_to_many",
    )
    accepted_with_grammy = pd.concat(
        [accepted_with_grammy.loc[~missing_id], recovered_ids],
        ignore_index=True,
        sort=False,
    )
accepted_with_grammy["_grammy_id"] = accepted_with_grammy["_grammy_id"].astype(int)

accepted_pairs = accepted_with_grammy[["_grammy_id", "track_id"]].drop_duplicates()
exact_pairs_for_filter = exact_matches[["_grammy_id", "track_id"]].drop_duplicates()
accepted_pairs = accepted_pairs.merge(
    exact_pairs_for_filter.assign(_already_exact=True),
    on=["_grammy_id", "track_id"],
    how="left",
)
accepted_pairs = accepted_pairs.loc[
    accepted_pairs["_already_exact"].isna(), ["_grammy_id", "track_id"]
]
accepted_with_grammy = accepted_with_grammy.merge(
    accepted_pairs,
    on=["_grammy_id", "track_id"],
    how="inner",
)

# Reattach canonical Spotify source fields and avoid ambiguous duplicate column names.
spotify_final_columns = [
    column for column in spotify_tracks.columns
    if column not in {"track_id", "_title_key", "_artist_keys"}
]
accepted_final = accepted_with_grammy.drop(
    columns=[column for column in spotify_final_columns if column in accepted_with_grammy.columns],
    errors="ignore",
).merge(
    spotify_tracks[["track_id", *spotify_final_columns]],
    on="track_id",
    how="left",
    validate="many_to_one",
)
accepted_final["match_method"] = accepted_final["_accepted_automatically"].map(
    {True: "FUZZY_AUTO_ACCEPTED", False: "MANUALLY_ACCEPTED_FUZZY"}
)
accepted_final["decision_manual"] = accepted_final["_accepted_automatically"].map(
    {True: "AUTO_ACCEPTED", False: "ACEPTAR"}
)

grammys_spotify_final = pd.concat(
    [exact_final, accepted_final],
    ignore_index=True,
    sort=False,
).drop_duplicates(["_grammy_id", "track_id"], keep="first")
matched_grammy_ids = set(grammys_spotify_final["_grammy_id"].dropna().unique())
unmatched_final = grammy_records.loc[
    ~grammy_records["_grammy_id"].isin(matched_grammy_ids),
    ["year", "title", "category", "nominee", "artist_resolved"],
].copy()

duplicate_pairs = grammys_spotify_final.duplicated(["_grammy_id", "track_id"]).sum()
assert duplicate_pairs == 0, "Duplicate Grammy-record/Spotify-track pairs remain in the final merge."

output_path = project_root / "data" / "interim" / "grammys_spotify_merged.csv"
unmatched_path = project_root / "data" / "interim" / "grammy_records_without_spotify_match.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
grammys_spotify_final.drop(
    columns=["_grammy_id", "_title_key", "_artist_keys"], errors="ignore"
).to_csv(output_path, index=False)
unmatched_final.to_csv(unmatched_path, index=False)

auto_accepted_count = accepted_final.loc[
    accepted_final["_accepted_automatically"], "_grammy_id"
].nunique()
manual_accepted_count = accepted_final.loc[
    ~accepted_final["_accepted_automatically"], "_grammy_id"
].nunique()
print("=== STAGE 3: FINAL MERGE ===")
print(f"Grammy source records: {len(grammy_records):,}")
print(f"Exact-matched Grammy records: {len(exact_grammy_ids):,}")
print(f"Additional Grammy records accepted automatically: {auto_accepted_count:,}")
print(f"Additional Grammy records accepted manually: {manual_accepted_count:,}")
print(f"Unique Grammy records matched in final output: {len(matched_grammy_ids):,}")
print(f"Final Spotify-linked rows: {len(grammys_spotify_final):,}")
print(f"Still without a Spotify match: {len(unmatched_final):,}")
print(f"Final coverage: {len(matched_grammy_ids) / len(grammy_records):.1%}")
print(f"\nFinal merged dataset: {output_path}")
print(f"Unmatched Grammy audit: {unmatched_path}")
display(grammys_spotify_final.head())

=== STAGE 3: FINAL MERGE ===
Grammy source records: 2,163
Exact-matched Grammy records: 241
Additional Grammy records accepted automatically: 43
Additional Grammy records accepted manually: 7
Unique Grammy records matched in final output: 289
Final Spotify-linked rows: 455
Still without a Spotify match: 1,874
Final coverage: 13.4%

Final merged dataset: /home/sebastiancelorio.dev/Projects/data-engineering/spotify-grammys-etl/data/interim/grammys_spotify_merged.csv
Unmatched Grammy audit: /home/sebastiancelorio.dev/Projects/data-engineering/spotify-grammys-etl/data/interim/grammy_records_without_spotify_match.csv


,year,title,category,nominee,artist_resolved,_grammy_id,_artist_keys_grammy,track_id,artists,album_name,...,match_score,candidate_rank,score_margin,decision_sugerida,nota_automatica,decision_manual,notas_revision,review_id,decision_sugerida_linkage,_accepted_automatically
0,2019,62nd Annual GRAMMY Awards (2019),Record Of The Year,Bad Guy,Billie Eilish,0,[billie eilish],2Fxmhks0bxGSBdJ92vM42m,Billie Eilish,"WHEN WE ALL FALL ASLEEP, WHERE DO WE GO?",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019,62nd Annual GRAMMY Awards (2019),Record Of The Year,Bad Guy,Billie Eilish,0,[billie eilish],5KJaEu2ZMIt9FMvofqUgmg,Billie Eilish,Отново на училище,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019,62nd Annual GRAMMY Awards (2019),Record Of The Year,Bad Guy,Billie Eilish,0,[billie eilish],154lbtHSyyOt0ao2QwbE4n,Billie Eilish,Girls Night,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019,62nd Annual GRAMMY Awards (2019),Record Of The Year,Bad Guy,Billie Eilish,0,[billie eilish],2yNKDG4FgwzQuL0JUEOunR,Billie Eilish,Herbststimmung,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019,62nd Annual GRAMMY Awards (2019),Record Of The Year,Bad Guy,Billie Eilish,0,[billie eilish],6DBdYfd4EkR6cUPSqtEwhQ,Billie Eilish,Beginner Ride (Sweat & Burn),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd

# Display all rows and columns.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

display(spotify_tracks)